In [2]:
from PIL import Image, ImageOps
import os
import numpy as np
import cv2

In [3]:
target_size = (512, 512)
scale_factor = 4

hr_output_folder = "/kaggle/working/data/high_resolution/"
lr_output_folder = "/kaggle/working/data/low_resolution/"
os.makedirs(hr_output_folder, exist_ok=True)
os.makedirs(lr_output_folder, exist_ok=True)

In [4]:
def resize_and_pad(img, target_size, pad_color=(0, 0, 0)):
    return ImageOps.pad(img, target_size, method=Image.BICUBIC, color=pad_color)

def downsample_image(hr_img, scale_factor):
    img_np = np.array(hr_img)
    
    num_steps = int(np.log2(scale_factor))  # For scale=4 → 2 steps

    for _ in range(num_steps):
        # Blur
        img_np = cv2.GaussianBlur(img_np, (3, 3), sigmaX=1)

        # Downsample by removing half of rows and columns
        img_np = img_np[::2, ::2]

    return Image.fromarray(img_np)
    

In [5]:
def process_folder(input_folder):
    # Process all images
    for filename in os.listdir(input_folder):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            img_path = os.path.join(input_folder, filename)
            img = Image.open(img_path).convert("RGB")
    
            # Resize & pad to 2K
            hr_img_2k = resize_and_pad(img, target_size)
            hr_img_2k.save(os.path.join(hr_output_folder, filename))
    
            # Generate low-res version
            lr_img = downsample_image(hr_img_2k, scale_factor)
            lr_img.save(os.path.join(lr_output_folder, filename))
    
    print("✅ All images resized to 2K and LR images generated.")

In [6]:
input_folder = "/kaggle/input/div2k-high-resolution-images/DIV2K_train_HR/DIV2K_train_HR"
process_folder(input_folder)
input_folder = "/kaggle/input/div2k-high-resolution-images/DIV2K_valid_HR/DIV2K_valid_HR"
process_folder(input_folder)
# input_folder = "/kaggle/input/flickr2k/Flickr2K"
# process_folder(input_folder)

✅ All images resized to 2K and LR images generated.
✅ All images resized to 2K and LR images generated.


In [7]:
import shutil

# Replace 'folder_name' with your actual folder
shutil.make_archive('cv_project', 'zip', '/kaggle/working/data')

'/kaggle/working/cv_project.zip'